# Project 67 — Local Structured Output Reliability Test

**Compare JSON adherence across prompts and models — locally with Ollama.**

| Component | Choice |
|-----------|--------|
| LLM | Ollama (qwen3:8b) |
| Validation | Pydantic |
| Interface | Jupyter |

## 1 · What You Will Learn

1. Measure **JSON output reliability** of local LLMs
2. Compare **prompt strategies** for structured output
3. Validate outputs against **Pydantic schemas**
4. Compute **parse success rate** and field completeness

## 2 · Why This Project Matters

Agents depend on structured JSON output. Models often return malformed JSON.
Measuring parse success rate is critical before deploying tool-calling pipelines.

## 3 · Setup

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## 4 · Imports

In [ ]:
import json, time
import pandas as pd
from pydantic import BaseModel, Field
from typing import Optional
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = 'qwen3:8b'
llm = ChatOllama(model=MODEL, temperature=0.1)
print(f'LLM ready: {MODEL}')

## 5 · Define Schemas

In [ ]:
class PersonExtraction(BaseModel):
    name: str
    age: Optional[int] = None
    occupation: Optional[str] = None
    location: Optional[str] = None

class SentimentResult(BaseModel):
    sentiment: str
    confidence: float
    reasoning: str

SCHEMAS = {
    'person': {'model': PersonExtraction, 'schema': PersonExtraction.model_json_schema()},
    'sentiment': {'model': SentimentResult, 'schema': SentimentResult.model_json_schema()},
}
print(f'{len(SCHEMAS)} schemas')

## 6 · Test Cases and Prompt Strategies

In [ ]:
TEST_INPUTS = {
    'person': ['John Smith, 34, software engineer in Austin.', 'Dr. Maria Garcia at Stanford.'],
    'sentiment': ['Amazing product, best purchase ever!', 'Terrible experience, never again.'],
}
STRATEGIES = {
    'bare': 'Return JSON matching: {schema}\nInput: {input}',
    'strict': 'Respond with ONLY valid JSON. No markdown.\nSchema: {schema}\nInput: {input}',
}
print(f'{sum(len(v) for v in TEST_INPUTS.values())} inputs, {len(STRATEGIES)} strategies')

## 7 · Run Tests

In [ ]:
def try_parse(raw):
    try: return json.loads(raw.strip()), True
    except: pass
    s, e = raw.find('{'), raw.rfind('}') + 1
    if s >= 0 and e > s:
        try: return json.loads(raw[s:e]), True
        except: pass
    return None, False

results = []
for schema_name, inputs in TEST_INPUTS.items():
    info = SCHEMAS[schema_name]
    for inp in inputs:
        for sname, stempl in STRATEGIES.items():
            prompt = ChatPromptTemplate.from_messages([('human', stempl)])
            t0 = time.time()
            raw = (prompt | llm | StrOutputParser()).invoke({'schema': json.dumps(info['schema']), 'input': inp})
            parsed, ok = try_parse(raw)
            valid = False
            if ok and parsed:
                try: info['model'](**parsed); valid = True
                except: pass
            results.append({'schema': schema_name, 'strategy': sname, 'input': inp[:30],
                            'parse_ok': ok, 'valid': valid, 'latency': round(time.time()-t0, 2)})
            print(f'  [{"PASS" if valid else "FAIL"}] {schema_name}/{sname}: {inp[:30]}')
print(f'Total: {len(results)}')

## 8 · Results

In [ ]:
df = pd.DataFrame(results)
print('PARSE SUCCESS BY STRATEGY:')
print(df.groupby('strategy')[['parse_ok','valid']].mean().round(3).to_string())
print(f'\nOverall validity: {df["valid"].mean():.1%}')

## 9 · Save

In [ ]:
df.to_csv('structured_output_results.csv', index=False)
print('Saved.')

## 10 · Key Takeaways

- Strict prompts improve parse success rate
- Pydantic validation catches type mismatches
- Robust JSON extraction from messy output is essential
- Local benchmarking reveals model-specific quirks